# IMPORT

In [3]:
import pandas as pd
from pathlib import Path
import joblib
import numpy as np
import matplotlib.pyplot as plt

import os
import gc
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

# DEFINE FOLDERS

In [4]:
BASE_FOLDER = Path("../data")
STATION_CLUSTER_FOLDER = BASE_FOLDER / "03_clustering" / "station_clusters"
DATASET_FOLDER = BASE_FOLDER / "01_clean" / "datasets"
MODEL_OUTPUT_FOLDER = BASE_FOLDER / "04_models" / "trained_models"
PLOT_FOLDER = BASE_FOLDER / "05_visualization" / "model_plots"
STATISTICS_FOLDER = BASE_FOLDER / "06_statistics" / "model_statistics"
MODEL_TEST_PLOTS_FOLDER = BASE_FOLDER / "model_test_plots"

folders = [STATION_CLUSTER_FOLDER, DATASET_FOLDER, MODEL_OUTPUT_FOLDER, PLOT_FOLDER, STATISTICS_FOLDER, MODEL_TEST_PLOTS_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# DEFINE CONSTANTS

In [12]:
ALL_FEATURES = True

RANDOM_STATE = 42 # 42 or 72
K = 2 # 2, 3, 4, 5
CLUSTER_METHOD = "kmeans"
FEATURE_NAME = "behavior+weather" if ALL_FEATURES else "behavior"
FILE_TAG = f"{CLUSTER_METHOD}_k{K}_feat_{FEATURE_NAME}_rs{RANDOM_STATE}"

CLUSTER_FILE_NAME = f"station_clusters_{FILE_TAG}"

HAS_BIKE_LAG = True
LAGGED_NAME = "lagged" if HAS_BIKE_LAG else "not_lagged"

MODEL_EXTENSION_TAG = f"feat_{FEATURE_NAME}_lag_{LAGGED_NAME}_rs{RANDOM_STATE}_k{K}"

# LOAD DATA AND CLUSTERS

In [22]:
df = pd.read_csv(f"{DATASET_FOLDER}/final_clean_split_data.csv")
clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/{CLUSTER_FILE_NAME}.csv")

df = df.merge(clusters, on="station_code", how="inner")

print("Rows:", df.shape)
print("Clusters:", df["cluster"].value_counts())
print(df.columns)

Rows: (13951711, 21)
Clusters: cluster
0    7030032
1    6921679
Name: count, dtype: int64
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'split', 'total_bikes_lag1',
       'total_bikes_lag2', 'cluster'],
      dtype='object')


# FEATURE SELECTION

In [15]:
FEATURES = [
    # Time
    "sin_hour",
    "cos_hour",
    "is_weekend",

    # Station context
    "capacity"
]

if ALL_FEATURES:
    FEATURES = FEATURES + [
                            # Weather
                            "temperature",
                            "wind_speed",
                            "wind_direction",
                            "weather_code",
                            "is_day",
                        ]
    
if HAS_BIKE_LAG:
    FEATURES = FEATURES + [
                            # Bike lag
                            "total_bikes_lag1", 
                            "total_bikes_lag2"
                        ]


TARGET = "total_bikes_now"


# TRAIN MODELS PER CLUSTER

In [6]:
pred_models = {}
results = []

for cluster_id in sorted(df["cluster"].unique()):
    print(f"- Training cluster {cluster_id}")

    # Select cluster and sort by time
    df_c = df[df["cluster"] == cluster_id].sort_values("timestamp")

    if HAS_BIKE_LAG:
        if "total_bikes_lag1" not in df_c.columns:
            df_c["total_bikes_lag1"] = df_c.groupby("station_code")["total_bikes_now"].shift(1)
            df_c["total_bikes_lag2"] = df_c.groupby("station_code")["total_bikes_now"].shift(2)
            df_c = df_c.dropna(subset=["total_bikes_lag1", "total_bikes_lag2"])

    # Select train/val using the split column
    train = df_c[df_c["split"] == 0]
    val   = df_c[df_c["split"] == 1]

    X_train, y_train = train[FEATURES], train[TARGET]
    X_val, y_val     = val[FEATURES], val[TARGET]

    models = {
        # "LinearRegression": LinearRegression(),
        # "SGDRegressor": Pipeline([
        #     ("scaler", StandardScaler()),
        #     ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        # ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ),
    }

    pred_models[cluster_id] = {}

    # Train and evaluate on validation set
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        r2 = r2_score(y_val, preds)
        mae = mean_absolute_error(y_val, preds)
        mse = mean_squared_error(y_val, preds)
        rmse = np.sqrt(mse)
        sse = mse * len(y_val)


        pred_models[cluster_id][name] = model

        results.append({
            "cluster": cluster_id,
            "model": name,
            "r2": r2,
            "mae": mae,
            "mse": mse,
            "rmse": rmse,
            "sse": sse,
            "n_samples": len(df_c)
        })

        print(f"  {name:16s} -> R^2 = {r2:.3f} | MAE = {mae:.2f} | MSE={mse:.2f} | RMSE={rmse:.2f} | SSE = {sse:.2f}")

        # Feature importance
        importances = None
        if name == "RandomForest":
            importances = model.feature_importances_
        elif name in ["LinearRegression", "SGDRegressor"]:
            # If pipeline, extract last step coefficients
            if isinstance(model, Pipeline):
                coef = model.named_steps["sgd"].coef_
            else:
                coef = model.coef_
            importances = np.abs(coef)
        else:
            importances = None

        if importances is not None:
            feat_imp_df = pd.DataFrame({
                "feature": FEATURES,
                "importance": importances
            }).sort_values("importance", ascending=False)

            # Plot
            plt.figure(figsize=(8, 5))
            plt.barh(feat_imp_df["feature"], feat_imp_df["importance"])
            plt.gca().invert_yaxis()
            plt.title(f"Feature Importance | Cluster {cluster_id} | {name} | {FEATURE_NAME} | {LAGGED_NAME}")
            plt.tight_layout()
            folder = Path(f"{PLOT_FOLDER}/{LAGGED_NAME}/{FEATURE_NAME}")
            folder.mkdir(parents=True, exist_ok=True)
            plt.savefig(f"{folder}/feature_importance_cluster_{cluster_id}_{name}.png")
            plt.close()
        else:
            print("NO PLOT!!!!")


- Training cluster 0
  RandomForest     -> R^2 = 0.134 | MAE = 4.62 | MSE=35.83 | RMSE=5.99 | SSE = 37760760.50
- Training cluster 1
  RandomForest     -> R^2 = 0.432 | MAE = 7.54 | MSE=88.53 | RMSE=9.41 | SSE = 91862707.81


# COMPARE RESULTS

In [7]:
results_df = pd.DataFrame(results)
results_df.sort_values(["cluster", "r2"], ascending=[True, False])

,cluster,model,r2,mae,mse,rmse,sse,n_samples
0,0,RandomForest,0.13441,4.615400,35.830976,5.985898,3.776076e+07,7030032
1,1,RandomForest,0.43185,7.539364,88.534006,9.409251,9.186271e+07,6921679


# SAVE RESULTS

In [8]:
results_df.to_csv(f"{STATISTICS_FOLDER}/training_results_{ALL_FEATURES=}_{HAS_BIKE_LAG=}.csv", index=False)

# SAVE ALL MODELS

In [9]:
for cluster_id, models_dict in pred_models.items():
    for name, model in models_dict.items():
        joblib.dump(model, f"{MODEL_OUTPUT_FOLDER}/{name}_model_cluster_{cluster_id}_{MODEL_EXTENSION_TAG}.pkl")

# QUICK TEST ON THE MODELS

In [1]:
def get_is_lagged(filename):
    return "not_lagged" not in filename

def get_has_behavior_weather(filename):
    return "behavior+weather" in filename

def get_features(filename):
    features = [
        # Time
        "sin_hour",
        "cos_hour",
        "is_weekend",

        # Station context
        "capacity"
    ]

    if get_has_behavior_weather(filename):
        features = features + [
                                # Weather
                                "temperature",
                                "wind_speed",
                                "wind_direction",
                                "weather_code",
                                "is_day",
                            ]
        
    if get_is_lagged(filename):
        features = features + [
                                # Bike lag
                                "total_bikes_lag1", 
                                "total_bikes_lag2"
                            ]
    return features

In [8]:
# LOAD THE MODELS
pred_models = {}

extension_tags = [
    f"feat_{fn}_lag_{lname}_rs{rs}_k{k}" 
    for fn in ["behavior+weather", "behavior"]
    for lname in ["lagged", "not_lagged"]
    for rs in [42]#[42, 72]
    for k in [2]
]

for cluster_id in range(2):
    pred_models[cluster_id] = {}
    for model_name in ["RandomForest"]:#["LinearRegression", "SGDRegressor", "RandomForest"]:
        pred_models[cluster_id][model_name] = []
        for extension_tag in extension_tags:
            filename = f"{MODEL_OUTPUT_FOLDER}/{model_name}_model_cluster_{cluster_id}_{extension_tag}.pkl"
            if os.path.exists(filename):
                pred_models[cluster_id][model_name].append({
                    "model": joblib.load(filename),
                    "is_lagged": get_is_lagged(filename),
                    "has_behavior_and_weather": get_has_behavior_weather(filename),
                    "features": get_features(extension_tag)
                    })
                
                print(f"Loaded: {filename}")
            else:
                print(f"File not found: {filename}")

Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_0_feat_behavior+weather_lag_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_0_feat_behavior+weather_lag_not_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_0_feat_behavior_lag_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_0_feat_behavior_lag_not_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_1_feat_behavior+weather_lag_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_1_feat_behavior+weather_lag_not_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_1_feat_behavior_lag_lagged_rs42_k2.pkl
Loaded: ..\data\04_models\trained_models/RandomForest_model_cluster_1_feat_behavior_lag_not_lagged_rs42_k2.pkl


# LOAD TEST DATA

In [9]:
test_dataset = pd.read_csv(f"{DATASET_FOLDER}/final_clean_split_data.csv")
test_dataset = test_dataset[test_dataset["split"] == 2].copy(deep=True)

In [16]:
from itertools import product

# Define all combinations
lag_options = [True, False]
feature_options = [True, False]

all_results = []

for HAS_BIKE_LAG, ALL_FEATURES in product(lag_options, feature_options):
    print(f"\nTesting: HAS_BIKE_LAG={HAS_BIKE_LAG}, ALL_FEATURES={ALL_FEATURES}")

    # Update constants for this combination
    LAGGED_NAME = "lagged" if HAS_BIKE_LAG else "not_lagged"
    FEATURE_NAME = "behavior+weather" if ALL_FEATURES else "behavior"
    FILE_TAG = f"{CLUSTER_METHOD}_k{K}_feat_{FEATURE_NAME}_rs{RANDOM_STATE}"
    CLUSTER_FILE_NAME = f"station_clusters_{FILE_TAG}"
    MODEL_EXTENSION_TAG = f"feat_{FEATURE_NAME}_lag_{LAGGED_NAME}_rs{RANDOM_STATE}_k{K}"

    # Load proper cluster file
    clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/{CLUSTER_FILE_NAME}.csv")

    # Prepare test dataset
    df_test = test_dataset.copy(deep=True)

    # Add lag columns if needed
    if HAS_BIKE_LAG:
        df_test["total_bikes_lag1"] = df_test.groupby("station_code")["total_bikes_now"].shift(1)
        df_test["total_bikes_lag2"] = df_test.groupby("station_code")["total_bikes_now"].shift(2)
        df_test = df_test.dropna(subset=["total_bikes_lag1", "total_bikes_lag2"])

    # Merge with clusters
    df_test = df_test.merge(clusters, on="station_code", how="inner")
    print(f"Test dataset shape: {df_test.shape}, clusters: {df_test['cluster'].value_counts()}")

    # Load models for this combination
    pred_models = {}
    extension_tag = MODEL_EXTENSION_TAG
    for cluster_id in df_test["cluster"].unique():
        pred_models[cluster_id] = {}
        for model_name in ["RandomForest"]:# ["LinearRegression", "SGDRegressor", "RandomForest"]:
            filename = f"{MODEL_OUTPUT_FOLDER}/{model_name}_model_cluster_{cluster_id}_{extension_tag}.pkl"
            if os.path.exists(filename):
                pred_models[cluster_id][model_name] = joblib.load(filename)
            else:
                print(f"Model not found: {filename}")

    # Test each cluster
    cluster_results = []
    for cluster_id in sorted(df_test["cluster"].unique()):
        df_c = df_test[df_test["cluster"] == cluster_id].sort_values("timestamp")
        y_val_full = df_c[TARGET]

        for model_name, model in pred_models[cluster_id].items():
            # Get proper features
            features = [
                "sin_hour", "cos_hour", "is_weekend", "capacity"
            ]
            if ALL_FEATURES:
                features += ["temperature","wind_speed","wind_direction","weather_code","is_day"]
            if HAS_BIKE_LAG:
                features += ["total_bikes_lag1","total_bikes_lag2"]

            X_val = df_c[features]
            y_val = y_val_full.loc[X_val.index]

            preds = model.predict(X_val)

            r2 = r2_score(y_val, preds)
            mae = mean_absolute_error(y_val, preds)
            mse = mean_squared_error(y_val, preds)
            rmse = np.sqrt(mse)
            sse = mse * len(y_val)

            cluster_results.append({
                "cluster": cluster_id,
                "model": model_name,
                "features": "+".join(features),
                "has_lag": HAS_BIKE_LAG,
                "all_features": ALL_FEATURES,
                "r2": r2,
                "mae": mae,
                "mse": mse,
                "rmse": rmse,
                "sse": sse,
                "n_samples": len(y_val)
            })

            print(f"Cluster {cluster_id} | {model_name} | R2={r2:.3f} | MAE={mae:.2f} | RMSE={rmse:.2f}")

    # Save results per combination
    combo_results_df = pd.DataFrame(cluster_results)
    combo_results_df.to_csv(
        STATISTICS_FOLDER / f"test_results_HAS_LAG_{HAS_BIKE_LAG}_ALL_FEATURES_{ALL_FEATURES}.csv",
        index=False
    )
    all_results.append(combo_results_df)

# Save overall combined results
all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.to_csv(STATISTICS_FOLDER / "test_results_all_combinations.csv", index=False)
print("Saved overall results to:", STATISTICS_FOLDER / "test_results_all_combinations.csv")



Testing: HAS_BIKE_LAG=True, ALL_FEATURES=True
Test dataset shape: (2096028, 21), clusters: cluster
0    1095182
1    1000846
Name: count, dtype: int64
Cluster 0 | RandomForest | R2=0.957 | MAE=0.84 | RMSE=1.49
Cluster 1 | RandomForest | R2=0.978 | MAE=1.00 | RMSE=1.79

Testing: HAS_BIKE_LAG=True, ALL_FEATURES=False
Test dataset shape: (2088658, 21), clusters: cluster
0    1052436
1    1036222
Name: count, dtype: int64
Cluster 0 | RandomForest | R2=0.955 | MAE=0.82 | RMSE=1.48
Cluster 1 | RandomForest | R2=0.978 | MAE=1.00 | RMSE=1.79

Testing: HAS_BIKE_LAG=False, ALL_FEATURES=True
Test dataset shape: (2098872, 21), clusters: cluster
0    1096668
1    1002204
Name: count, dtype: int64
Cluster 0 | RandomForest | R2=0.150 | MAE=4.83 | RMSE=6.64
Cluster 1 | RandomForest | R2=0.420 | MAE=7.29 | RMSE=9.21

Testing: HAS_BIKE_LAG=False, ALL_FEATURES=False
Test dataset shape: (2091492, 21), clusters: cluster
0    1053864
1    1037628
Name: count, dtype: int64
Cluster 0 | RandomForest | R2=0.18

In [17]:
display(all_results)

[   cluster         model                                           features  \
 0        0  RandomForest  sin_hour+cos_hour+is_weekend+capacity+temperat...   
 1        1  RandomForest  sin_hour+cos_hour+is_weekend+capacity+temperat...   
 
    has_lag  all_features        r2       mae       mse      rmse  \
 0     True          True  0.957088  0.837657  2.226452  1.492130   
 1     True          True  0.977971  0.995712  3.221501  1.794854   
 
             sse  n_samples  
 0  2.438370e+06    1095182  
 1  3.224227e+06    1000846  ,
    cluster         model                                           features  \
 0        0  RandomForest  sin_hour+cos_hour+is_weekend+capacity+total_bi...   
 1        1  RandomForest  sin_hour+cos_hour+is_weekend+capacity+total_bi...   
 
    has_lag  all_features        r2       mae       mse      rmse  \
 0     True         False  0.955291  0.822064  2.190331  1.479977   
 1     True         False  0.978291  1.000726  3.214585  1.792926   
 
       

In [18]:
results = pd.DataFrame(all_results)
results.sort_values(["cluster", "r2"], ascending=[True, False])

ValueError: Must pass 2-d input. shape=(4, 2, 11)

# PLOT IMPORTANCE OF MODELS

In [19]:
def plot_feature_importance(pred_models):
    for cluster_id in range(2):
        for model_name, versions in pred_models[cluster_id].items():
            for version_index, model_entry in enumerate(versions):
                model = model_entry["model"]
                is_lagged = model_entry["is_lagged"]
                has_behavior_and_weather = model_entry["has_behavior_and_weather"]
                features = model_entry["features"]
                lagged_name = "lagged" if is_lagged else "not_lagged"
                feature_name = "behavior+weather" if has_behavior_and_weather else "behavior"

                # Feature importance
                importances = None
                if model_name == "RandomForest":
                    importances = model.feature_importances_
                elif model_name in ["LinearRegression", "SGDRegressor"]:
                    # If pipeline, extract last step coefficients
                    if isinstance(model, Pipeline):
                        coef = model.named_steps["sgd"].coef_
                    else:
                        coef = model.coef_
                    importances = np.abs(coef)
                else:
                    importances = None

                if importances is not None:
                    feat_imp_df = pd.DataFrame({
                        "feature": features,
                        "importance": importances
                    }).sort_values("importance", ascending=False)

                    out_dir = PLOT_FOLDER / lagged_name / feature_name
                    out_dir.mkdir(parents=True, exist_ok=True)
                    # Plot
                    plt.figure(figsize=(8, 5))
                    plt.barh(feat_imp_df["feature"], feat_imp_df["importance"])
                    plt.gca().invert_yaxis()
                    plt.title(f"Feature Importance | Cluster {cluster_id} | {model_name} | {feature_name} | {lagged_name}")
                    plt.tight_layout()
                    plt.savefig(f"{PLOT_FOLDER}/feature_importance_cluster_{cluster_id}_{model_name}_v{version_index}.png")
                    plt.savefig(f"{out_dir}/feature_importance_cluster_{cluster_id}_{model_name}_v{version_index}.png")
                    plt.close()
                else:
                    print("NO PLOT!!!!")

In [20]:
plot_feature_importance(pred_models)

TypeError: 'DecisionTreeRegressor' object is not subscriptable

In [23]:
df_val = df[df["split"] == 1].copy()
del df
gc.collect()

3010

# VALIDATE

In [24]:
def plot_station(args):
    cluster_id, model_name, version_index, station_code, df_station = args
    
    model_entry = pred_models[cluster_id][model_name][version_index]
    model = model_entry["model"]
    features = model_entry["features"]
    is_lagged = model_entry["is_lagged"]
    has_behavior_and_weather = model_entry["has_behavior_and_weather"]

    df_station = df_station.sort_values("timestamp")

    X = df_station[features]
    y = df_station[TARGET]

    preds = model.predict(X)

    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    mse = mean_squared_error(y, preds)
    rmse = np.sqrt(mse)
    sse = mse * len(y)


    plt.figure(figsize=(14, 5))
    plt.plot(df_station["timestamp"], y, label="Actual", alpha=0.8)
    plt.plot(df_station["timestamp"], preds, label="Predicted", alpha=0.8)

    plt.title(
        f"Cluster {cluster_id} | Station {station_code}\n"
        f"Model: {model_name}"
    )
    plt.xlabel("Timestamp")
    plt.ylabel("Total Bikes Now")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        f"{PLOT_FOLDER}/cluster_{cluster_id}_station_{station_code}_{FEATURE_NAME}_{LAGGED_NAME}.png"
    )
    plt.close()


    return {
        "cluster": cluster_id,
        "station": station_code,
        "model": model_name,
        "is_lagged": is_lagged,
        "has_behavior_and_weather": has_behavior_and_weather,
        "r2": r2,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "sse": sse,
        "n_samples": len(df_station),
    }


In [25]:
os.cpu_count()
gc.collect()

0

In [26]:
jobs = [
    (cluster_id, model_name, version_index, station_code, df_station)
    for (cluster_id, station_code), df_station in df_val.groupby(["cluster", "station_code"], sort=False)
    for model_name, version_index in pred_models[cluster_id].items()
    for version_index in range(len(pred_models[cluster_id][model_name]))
]
results = []

with ThreadPoolExecutor(max_workers=6) as ex:
    for res in tqdm(ex.map(plot_station, jobs), total=len(jobs)):
        results.append(res)

results_df = pd.DataFrame(results)
results_df.to_csv(
    f"{STATISTICS_FOLDER}/all_stations_results_all_models.csv",
    index=False
)

  0%|          | 0/141700 [00:00<?, ?it/s]


TypeError: 'DecisionTreeRegressor' object is not subscriptable